In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q pyzipper gdown

import pyzipper
import gdown
import os
from unsloth import FastLanguageModel
from google.colab import drive
drive.mount('/content/drive')

# 1. Configuration
SHARED_LINK = "PASTE_YOUR_LINK_HERE" # Paste your Google Drive shared link here
local_zip_path = "/content/temp_adapter_encrypted.zip"
extract_dir = "/content/final_adapter"
password = "PASTE_YOUR_KEY_HERE" # Your AES key

# Helper to pull the exact file ID from a standard share link
def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# 2. Download from Shared Link
print("Downloading encrypted master backup from Google Drive...")
file_id = get_gdrive_id(SHARED_LINK)
download_url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(download_url, local_zip_path, quiet=False)

# 3. Decrypt and unzip
print("Decrypting master backup...")
os.makedirs(extract_dir, exist_ok=True)
with pyzipper.AESZipFile(local_zip_path, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zf:
    zf.pwd = password.encode('utf-8')
    zf.extractall(extract_dir)

# 4. Load the adapter back into Unsloth
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = extract_dir,
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

# 5. Export the GGUF directly to your Drive!
print("Compiling GGUF...")
save_dir = "/content/drive/MyDrive/soulclone"
os.makedirs(save_dir, exist_ok=True)

# Unsloth will append the quantization string and spit out the .gguf here
gguf_prefix = os.path.join(save_dir, "cpu_model_rescue")
model.save_pretrained_gguf(gguf_prefix, tokenizer, quantization_method="q4_k_m")

print(f"Done! Check your Google Drive at {save_dir} for the .gguf file.")

In [ ]:
import time
from google.colab import runtime

print("Pipeline complete! All files should be saved to Google Drive.")
print("The runtime will automatically disconnect and delete itself in 60 seconds to save compute units...")

# Wait 1 minute to ensure Google Drive I/O operations have fully synced
time.sleep(60)

print("Terminating runtime now.")
# This command forcefully disconnects and deletes the Colab instance
runtime.unassign()